# NVIDIA Smart City Blueprint — Quickstart Guide (Docker Compose)

**Source:** [NVIDIA VSS 3.1.0 Smart City Quickstart](https://docs.nvidia.com/vss/3.1.0/smartcity-docs/Quickstart-Guide.html)

This notebook walks through deploying the NVIDIA Smart City Blueprint using Docker Compose, based on the VSS 3.0 framework. It covers:
- Environment setup and NGC access
- Downloading sample data and models
- Configuring and deploying the blueprint
- Verifying the deployment
- Exploring the system (UI, alerts, Kibana, map)
- Teardown

> ⚠️ **Important:** Run cells in order. Most cells execute shell commands via `!` or `%%bash`. Ensure Docker, NGC CLI, and sufficient GPU resources are available before starting.

## Architecture Overview

The Smart City Blueprint uses a layered architecture based on VSS 3.0:

| Layer | Components |
|---|---|
| **Foundational Microservices** | Vision Language Models (VLMs), Computer Vision (CV) models |
| **Real-Time Video Processing** | Live camera feed pipelines via DeepStream / RTVI |
| **AI Agents** | Intelligent analysis, natural language interaction |

### Supported GPU / Stream Configurations

| GPU | Streams (RTDETR) | Streams (GDINO) | GPUs (Local VLM+LLM dedicated) | GPUs (shared) | GPUs (Remote) |
|---|---|---|---|---|---|
| H100 | 30 | 12 | 3 | 2 | 1 |
| L40S | 30 | 6 | 3 | N/A | 1 |
| RTX PRO 6000 Blackwell | 30 | 12 | 3 | 2 | 1 |

> Numbers based on default config: 1080p @ 10fps.

---
## Step 0 — Prerequisites Check

Verify that required tools are available on the host.

In [ ]:
import subprocess, sys

tools = ["docker", "ngc", "python3", "tar", "bash"]
all_ok = True

for tool in tools:
    result = subprocess.run(["which", tool], capture_output=True, text=True)
    status = "✅" if result.returncode == 0 else "❌ NOT FOUND"
    if result.returncode != 0:
        all_ok = False
    print(f"{status}  {tool}")

if not all_ok:
    print("\n⚠️  Some tools are missing. Install them before continuing.")
else:
    print("\n✅ All tools found.")

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null \
    || echo "❌ nvidia-smi not found or no GPU detected"

In [ ]:
# Check Docker daemon and NVIDIA runtime
!docker info 2>/dev/null | grep -E 'Runtimes|Server Version' || echo "❌ Docker daemon not running"

---
## Step 1 — Configure Environment Variables

Set your credentials and paths here. These are exported for all subsequent shell cells.

In [ ]:
import os

# ─── REQUIRED: fill these in ───────────────────────────────────────────────
NGC_CLI_API_KEY     = "YOUR_NGC_API_KEY_HERE"
GOOGLE_MAPS_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY_HERE"
VSS_DATA_DIR        = os.path.expanduser("~/vss-data")   # change as needed
HARDWARE_PROFILE    = "H100"   # One of: H100, L40S, RTXPRO6000BW
# ───────────────────────────────────────────────────────────────────────────

os.environ["NGC_CLI_API_KEY"]     = NGC_CLI_API_KEY
os.environ["NGC_CLI_ORG"]         = "nvidia"
os.environ["VSS_DATA_DIR"]        = VSS_DATA_DIR
os.environ["HARDWARE_PROFILE"]    = HARDWARE_PROFILE
os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY

print(f"NGC_CLI_API_KEY     = {'*' * len(NGC_CLI_API_KEY)}")
print(f"NGC_CLI_ORG         = nvidia")
print(f"VSS_DATA_DIR        = {VSS_DATA_DIR}")
print(f"HARDWARE_PROFILE    = {HARDWARE_PROFILE}")
print(f"GOOGLE_MAPS_API_KEY = {'*' * len(GOOGLE_MAPS_API_KEY)}")

---
## Step 2 — Create Data Directory

In [ ]:
%%bash
mkdir -p "${VSS_DATA_DIR}"
echo "Created: ${VSS_DATA_DIR}"
mkdir "${VSS_DATA_DIR}"/agent_eval
chmod -R 777 "${VSS_DATA_DIR}"/agent_eval
mkdir "${VSS_DATA_DIR}"/engines
chmod -R 777 "${VSS_DATA_DIR}"/engines

---
## Step 3 — Download Sample Data

Downloads the Smart City app data from NGC and extracts it.
Only `Agnew_head_on.mp4` is included by default; the other four collision clips are excluded.
See [Collision Clips Selection](#collision-clips) to choose a different clip.

In [ ]:
%%bash
set -e

echo "==> Downloading sample data from NGC..."
ngc registry resource download-version \
    nvidia/vss-smartcities/vss-smartcities-app-data:3.1.0

echo "==> Extracting (excluding secondary collision clips)..."
tar -xzf vss-smartcities-app-data_v3.1.0/smartcities-app-data.tar.gz \
    -C "${VSS_DATA_DIR}" \
    --strip-components=1 \
    --exclude='*/Agnew_left_turn.mp4' \
    --exclude='*/Agnew_rear_end.mp4' \
    --exclude='*/Agnew_sideswipe_1.mp4' \
    --exclude='*/Agnew_sideswipe_2.mp4'

echo "==> Setting permissions..."
chmod -R 777 "${VSS_DATA_DIR}/data_log"

echo "==> Cleaning up archive..."
rm -rf vss-smartcities-app-data_v3.1.0

echo "Done. Data directory contents:"
ls "${VSS_DATA_DIR}"

### Collision Clips Selection (Optional)

Available clips for the Agnew intersection:

| Clip | Notes |
|---|---|
| `Agnew_head_on.mp4` | **Default** — included above |
| `Agnew_left_turn.mp4` | Excluded above |
| `Agnew_rear_end.mp4` | Excluded above |
| `Agnew_sideswipe_1.mp4` | Excluded above |
| `Agnew_sideswipe_2.mp4` | Excluded above |

To switch clips, re-run the extraction omitting a different set and update the calibration file:

In [ ]:
# Uncomment and set the desired clip name to switch
# NEW_CLIP = "Agnew_sideswipe_2"
# CALIB = os.path.join(os.environ.get("MDX_SAMPLE_APPS_DIR", "."),
#                      "smartcities/smc-app/calibration/sample-data/calibration.json")
# !sed -i "s/Agnew_head_on/{NEW_CLIP}/g" "{CALIB}"
# print(f"Calibration updated to use: {NEW_CLIP}")

---
## Step 4 — Download Models

Downloads RTDETR and GDINO model weights from NGC.

In [ ]:
%%bash
set -e

echo "==> Creating model directories..."
mkdir -p "${VSS_DATA_DIR}/models/rtdetr-its"
mkdir -p "${VSS_DATA_DIR}/models/gdino"

echo "==> Downloading RTDETR model (TrafficCamNet Transformer)..."
ngc registry model download-version \
    nvidia/tao/trafficcamnet_transformer_lite:deployable_resnet50_v2.0

mv trafficcamnet_transformer_lite_vdeployable_resnet50_v2.0/resnet50_trafficcamnet_rtdetr.fp16.onnx \
   "${VSS_DATA_DIR}/models/rtdetr-its/model_epoch_035.fp16.onnx"

rm -rf trafficcamnet_transformer_lite_vdeployable_resnet50_v2.0
chown -R ubuntu:ubuntu  ${VSS_DATA_DIR}/models/rtdetr-its/
echo "RTDETR model ready: ${VSS_DATA_DIR}/models/rtdetr-its/"

In [ ]:
%%bash
set -e

echo "==> Downloading GDINO model (Mask Grounding DINO)..."
ngc registry model download-version \
    nvidia/tao/mask_grounding_dino:mask_grounding_dino_swin_tiny_commercial_deployable_v2.1_wo_mask_arm

mv mask_grounding_dino_vmask_grounding_dino_swin_tiny_commercial_deployable_v2.1_wo_mask_arm/mgdino_mask_head_pruned_dynamic_batch.onnx \
   "${VSS_DATA_DIR}/models/gdino/mgdino_mask_head_pruned_dynamic_batch.onnx"

rm -rf mask_grounding_dino_vmask_grounding_dino_swin_tiny_commercial_deployable_v2.1_wo_mask_arm

chmod -R 777 "${VSS_DATA_DIR}/models"
echo "GDINO model ready: ${VSS_DATA_DIR}/models/gdino/"
chown -R ubuntu:ubuntu  ${VSS_DATA_DIR}/models/gdino/
echo "\nAll models:"
find "${VSS_DATA_DIR}/models" -name '*.onnx'

---
## Step 5 — Download Compose Project

In [ ]:
%%bash
set -e
cd ~
git clone git@github.com:NVIDIA-AI-Blueprints/video-search-and-summarization.git
cd ~/video-search-and-summarization
git checkout develop

---
## Step 6 — Configure .env File

Merge smartcities/.env into dev-profile-alerts/.env.

In [ ]:
%%bash
set -e

cd ~/video-search-and-summarization/deploy/docker

export VSS_APPS_DIR="$(realpath .)"
#export HOST_IP="$(ip route get 1.1.1.1 | awk '/src/ {for (i=1;i<=NF;i++) if ($i=="src") print $(i+1)}')"

#echo "HOST_IP detected: ${HOST_IP}"
echo "VSS_APPS_DIR: ${VSS_APPS_DIR}"

sed -i \
    -e "s|^VSS_APPS_DIR=.*|VSS_APPS_DIR=\"${VSS_APPS_DIR}\"|" \
    -e "s|^VSS_DATA_DIR=.*|VSS_DATA_DIR=\"${VSS_DATA_DIR}\"|" \
    #-e "s|^HOST_IP=.*|HOST_IP=\"${HOST_IP}\"|" \
    -e "s|^NGC_CLI_API_KEY=.*|NGC_CLI_API_KEY=\"${NGC_CLI_API_KEY}\"|" \
    -e "s|^GOOGLE_MAPS_API_KEY=.*|GOOGLE_MAPS_API_KEY=\"${GOOGLE_MAPS_API_KEY}\"|" \
    ~/video-search-and-summarization/deploy/docker/industry-profiles/smartcities/.env

echo "==> .env updated. Current relevant values:"
grep -E '^(VSS_DATA_DIR|VSS_APPS_DIR|HARDWARE_PROFILE)=' ~/video-search-and-summarization/deploy/docker/industry-profiles/smartcities/.env

awk -F= 'ARGIND==1 { if ($0 !~ /^[[:space:]]*#/ && $0 ~ /=/) env[$1]=$0; next } ($1 in env) {print env[$1]; delete env[$1]; next} {print} END {for (k in env) print env[k]}' ~/video-search-and-summarization/deploy/docker/industry-profiles/smartcities/.env /home/ubuntu/tnoumessi/video-search-and-summarization/deploy/docker/developer-profiles/dev-profile-alerts/.env > /tmp/.target.env.tmp && cp /tmp/.target.env.tmp ~/video-search-and-summarization/deploy/docker/industry-profiles/smartcities/.env

# sync smartcities related config to alert verification env
rsync -av ~/video-search-and-summarization/deploy/docker/industry-profiles/smartcities/ ~/video-search-and-summarization/deploy/docker/developer-profiles/dev-profile-alerts/

# do not allow DATA and APP Dir to be updated
sed -i.bak -E -e '/^[[:space:]]*#/!{' -e '/VSS_APPS_DIR|VSS_DATA_DIR/ s/^/# /' -e '}' /Users/tnoumessi/Documents/git-project/video-search-and-summarization/deploy/docker/scripts/dev-profile.sh && grep -nE 'VSS_APPS_DIR|VSS_DATA_DIR' ~/git-project/video-search-and-summarization/deploy/docker/scripts/dev-profile.sh

---
## Step 7 — Custom VLM Weights (Optional)

Skip this section if using the default VLM. Run only if you have custom weights.

In [ ]:
# ── Option A: Download from NGC ──────────────────────────────────────────────
# Uncomment and set your values:

# VLM_CUSTOM_WEIGHTS = "/path/to/custom/weights"
# !ngc registry model download-version <org>/<team>/<model>:<version>
# !mv </downloaded/ngc/output> {VLM_CUSTOM_WEIGHTS}
# import os; os.environ["VLM_CUSTOM_WEIGHTS"] = VLM_CUSTOM_WEIGHTS
# !cd ~/vss-blueprints/deployments && \
#  sed -i -e "s|^# VLM_CUSTOM_WEIGHTS.*|VLM_CUSTOM_WEIGHTS=\"{VLM_CUSTOM_WEIGHTS}\"|" smartcities/.env
# print("Custom VLM weights set from NGC.")

print("(Skipped — using default VLM)")

In [ ]:
# ── Option B: Download from Hugging Face ─────────────────────────────────────
# Uncomment and set your values:

# HF_MODEL_ID         = "<org>/<model-name>"
# VLM_CUSTOM_WEIGHTS  = "/path/to/custom/weights"
#
# !pip install -U "huggingface_hub[cli]" -q
# !hf auth login
# !mkdir -p {VLM_CUSTOM_WEIGHTS}
# !hf download {HF_MODEL_ID} --local-dir {VLM_CUSTOM_WEIGHTS}

print("(Skipped — using default VLM)")

---
## Step 8 — Deploy the Blueprint

### 8a. Select Deployment Profile

The default `bp_smc` profile includes all features. LLM/VLM modes:

| Mode | Description |
|---|---|
| `local` | Deploy NIMs locally on dedicated GPUs (requires NGC key) |
| `local_shared` | LLM + VLM share a single GPU |
| `remote` | Use external NIM endpoints (requires `LLM_BASE_URL` + `VLM_BASE_URL`) |

To use remote NIMs, update `~/video-search-and-summarization/deploy/docker/industry-profiles/smartcities/.env` with `LLM_MODE=remote`, `VLM_MODE=remote`, and the endpoint URLs.

In [ ]:
# Configure remote NIMs (optional — skip for local mode)
#
# LLM_MODE = "remote"
# VLM_MODE = "remote"
# LLM_BASE_URL = "http://<remote-host>:<llm-port>"
# VLM_BASE_URL = "http://<remote-host>:<vlm-port>"
# LLM_NAME = "nvidia/llama-3.3-nemotron-super-49b-v1.5"
# VLM_NAME = "nvidia/cosmos-reason1-7b"
#
# Edit smartcities/.env directly or use sed commands here.
print("(Using default local mode)")

### 8b. Docker Login to NGC Registry

In [ ]:
%%bash
echo "==> Logging into NGC Docker registry..."
echo "${NGC_CLI_API_KEY}" | docker login \
    --username '$oauthtoken' \
    --password-stdin \
    nvcr.io && echo "Docker login successful."

### 8c. Start the Blueprint

> ⏳ First launch pulls large containers — this may take 20–60 minutes depending on network speed.

In [ ]:
%%bash
set -e
cd ~/video-search-and-summarization/deploy/docker
./scripts/dev-profile.sh up -p alerts -H H100 -m verification

# export VSS_APPS_DIR="$(realpath .)"
# echo "==> Starting Smart City Blueprint..."
# docker compose \
#     --env-file industry-profiles/smartcities/.env \
#     up \
#     --detach \
#     --force-recreate \
#     --build

echo "\n==> Deployment initiated. Waiting 120s for containers to start..."
sleep 120
docker compose ls

### 8d. Optional: Kibana Dashboard for Alert Verification

In [ ]:
%%bash
# Uncomment to set up Kibana VLM alerts dashboard:
# cd "${VSS_SAMPLE_APPS_DIR}/industry-profiles/smartcities/smc-app/kibana-dashboard/init-scripts"
# bash vlm-alerts-template-setup.sh
# cd "${VSS_SAMPLE_APPS_DIR}"
echo "(Kibana setup skipped — uncomment above to enable)"

---
## Step 9 — Verify Deployment

In [ ]:
%%bash
echo "=== Running containers ==="
docker ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'

echo "\n=== Compose stacks ==="
docker compose ls

In [ ]:
%%bash
# Check Perception FPS (should be ~10fps with sample streams)
echo "=== Perception logs (last 50 lines — look for FPS) ==="
docker logs --tail 50 vss-rtvi-cv 2>&1 | grep -E '(FPS|fps|Error|error)' || \
    echo "No FPS lines yet — container may still be initializing."

In [ ]:
import os
import subprocess
import urllib.request

# Detect HOST_IP
result = subprocess.run(
    "ip route get 1.1.1.1 | awk '/src/ {for(i=1;i<=NF;i++) if($i=="'src'") print $(i+1)}'",
    shell=True, capture_output=True, text=True
)
HOST_IP = result.stdout.strip() or "localhost"
print(f"HOST_IP: {HOST_IP}\n")

# VSS UI has Alert UI/Video Analytics UI(MAP)/Kibana UI
endpoints = {
    "VSS-UI":               f"http://{HOST_IP}:3000",
    "NvStreamer-UI":         f"http://{HOST_IP}:31000/#/dashboard",
    "VST-UI":               f"http://{HOST_IP}:30888/vst/#/dashboard",
    "Phoenix-UI":           f"http://{HOST_IP}:6006/projects",
    "Video-Analytics-API":  f"http://{HOST_IP}:8081",
    "VST-MCP":              f"http://{HOST_IP}:8001",
    "VA-MCP":               f"http://{HOST_IP}:9901",
    "LLM-NIM":              f"http://{HOST_IP}:30081",
    "VLM-NIM":              f"http://{HOST_IP}:30082",
}

print(f"{'Service':<25} {'URL':<45} Status")
print("-" * 85)
for name, url in endpoints.items():
    check_url = url.split('#')[0]  # strip anchor fragments
    try:
        req = urllib.request.urlopen(check_url, timeout=3)
        status = f"✅ {req.status}"
    except Exception as e:
        status = f"⏳ {str(e)[:35]}"
    print(f"{name:<25} {url:<45} {status}")

---
## Step 10 — Add Custom Streams (Optional)

The default deployment includes 4 pre-configured streams. Use the helper scripts to add more.

In [ ]:
# Upload videos to NVStreamer
# Modify the path and stream count before running:

# VIDEO_DIR   = "/path/to/your/videos/"
# NUM_STREAMS = 10
# !python3 ~/video-search-and-summarization/deploy/docker/scripts/nvstreamer_upload.py \
#     http://localhost:31000/nvstreamer/ {NUM_STREAMS} {VIDEO_DIR}

print("(Custom stream upload skipped — update VIDEO_DIR and NUM_STREAMS above)")

In [ ]:
# Add NVStreamer sensors to VST

# MAX_STREAMS = 4
# !python3 ~/video-search-and-summarization/deploy/docker/scripts/add-nvstreamer-to-vst.py \
#     --nvstreamer_endpoint=http://localhost:31000/nvstreamer \
#     --vst_endpoint=http://localhost:30888/vst/ \
#     --max_streams_to_add={MAX_STREAMS}

print("(VST sensor registration skipped — uncomment above to enable)")

In [ ]:
# Delete ALL sensors from VST (use with caution)

# !python3 ~/video-search-and-summarization/deploy/docker/scripts/add-nvstreamer-to-vst.py \
#     --vst_endpoint=http://localhost:30888/vst/ \
#     --delete_all=1

print("(Sensor deletion skipped — uncomment above if needed)")

---
## Step 11 — Explore the Blueprint

The UI is available at `http://<HOST_IP>:3000`. Features to explore:

### 💬 Chat Interface

Natural language interaction with the system. Example prompts:

| Category | Prompt |
|---|---|
| Sensor Discovery | `What sensors are available?` |
| Live Snapshot | `Can you take a snapshot of sensor [ID]?` |
| Recent Incidents | `List the last 5 incidents for [Sensor ID].` |
| Time Window | `List all incidents from [Start] to [End] for sensor [ID].` |
| By Location | `Retrieve all incidents for [Place Name].` |
| Report | `Generate a report for incident [Incident ID].` |
| Follow-up | `Generate a report for the second one.` |
| Pagination | `Show the next 20 incidents.` |

### 🚨 Alerts Tab

| Filter | Purpose |
|---|---|
| VLM Verified | Toggle to see alerts confirmed/rejected by VLM |
| Verdict | Filter: `confirmed`, `rejected`, or `verification failed` |
| Period | 10 min – 2 hours (or custom) |
| Sensor | Per-sensor filter |
| Alert Type | Currently: `collision` only |
| Alert Triggered | `Abnormal movement` or `Stop by speed` |

### 📊 Kibana Dashboard

Navigate to `http://<HOST_IP>:5601/` → Menu → Dashboard → **ITS Dashboard**

Charts available:
- **Sensor Table** — Object counts per sensor
- **Vehicle Speed** — Donut chart (0–15, 15–30, 30–45 mph...)
- **Vehicle Distance** — Distance distribution in meters
- **Direction** — N/S/E/W vehicle movement
- **Alerts Count** / **VLM Verified Alerts** — True vs. false positive breakdown

### 🗺️ Map View

Available at `http://<HOST_IP>:3002`

| Marker Color | Meaning |
|---|---|
| 🟢 Green | Active intersection (data flowing) |
| 🔵 Blue | Selected corridor |
| 🔴 Red | Recent severe alert |

Road segment colors indicate average vehicle speed (orange = slow, green = free-flowing).

In [ ]:
import subprocess

result = subprocess.run(
    "ip route get 1.1.1.1 | awk '/src/ {for(i=1;i<=NF;i++) if($i=="'src'") print $(i+1)}'",
    shell=True, capture_output=True, text=True
)
HOST_IP = result.stdout.strip() or "<HOST_IP>"

print("=" * 55)
print("  Smart City Blueprint — Endpoint Reference")
print("=" * 55)

ui_endpoints = [
    ("VSS-UI",              3000, ""),
    ("Video-Analytics-UI",  3002, ""),
    ("Grafana-UI",          35000, "/"),
    ("Kibana-UI",           5601,  "/app/home#/"),
    ("NvStreamer-UI",        31000, "/#/dashboard"),
    ("VST-UI",              30888, "/vst/#/dashboard"),
    ("Phoenix-UI",           6006,  "/projects"),
]
api_endpoints = [
    ("Video-Analytics-API", 8081,  ""),
    ("VST-MCP",             8001,  ""),
    ("VA-MCP",              9901,  ""),
    ("LLM-NIM",             30081, ""),
    ("VLM-NIM",             30082, ""),
]

print("\n  UI Endpoints:")
for name, port, path in ui_endpoints:
    print(f"  {name:<22} http://{HOST_IP}:{port}{path}")

print("\n  API Endpoints:")
for name, port, path in api_endpoints:
    print(f"  {name:<22} http://{HOST_IP}:{port}{path}")

---
## Step 12 — Teardown

> ⚠️ This will stop and remove all containers and dangling volumes. Data in `VSS_DATA_DIR` is preserved unless you run the cleanup script.

In [ ]:
%%bash
# Uncomment to teardown:
cd ~/video-search-and-summarization/deploy/docker

# echo "==> Stopping blueprint..."
docker compose -f industry-profiles/smartcities/compose.yml down

# echo "==> Removing dangling volumes..."
# docker volume ls -q -f "dangling=true" | xargs docker volume rm

# echo "==> Cleaning data logs..."
bash scripts/cleanup_all_datalog.sh -b smartcities

echo "(Teardown skipped — uncomment above when ready)"

---
## Known Issues & Limitations

| Issue | Notes |
|---|---|
| Shared GPU NIMs | Only the default NIMs are tested on shared GPU. Use dedicated or remote NIMs for other models. |
| WebRTC Video Playback | May fail if the service is behind a private network where the WebRTC TURN server is not accessible. |
| Spiderfied Map Markers | When a marker status changes while spiderfied, it may revert to its original overlapping position. |

---
## References

- [NVIDIA VSS 3.1.0 Smart City Quickstart](https://docs.nvidia.com/vss/3.1.0/smartcity-docs/Quickstart-Guide.html)
- [Smart City Prerequisites](https://docs.nvidia.com/vss/3.1.0/smartcity-docs/Prerequisites.html)
- [VSS Agents Overview](https://docs.nvidia.com/vss/3.1.0/VSS-Agents.html)
- [RTVI Microservice](https://docs.nvidia.com/vss/3.1.0/object-detection-tracking.html)
- [Alert Verification Microservice](https://docs.nvidia.com/vss/3.1.0/alert-verification-service.html)
- [Kibana Dashboard Docs](https://www.elastic.co/guide/en/kibana/current/dashboard.html)